# Анализ эффективности email-рассылок

## Цель исследования
Целью данного исследования является оценка эффективности email-рассылок на основе предоставленного тестового набора данных.  
В рамках анализа рассчитываются ключевые метрики email-маркетинга, а также определяется наиболее успешная тема письма.

## Описание данных
В работе используется тестовый набор данных, содержащий агрегированные результаты email-рассылок, включая:
- количество отправленных писем;
- количество доставленных писем;
- количество открытий;
- количество кликов;
- количество отписок.

Рассчитываются следующие показатели:

- коэффициент доставки (Delivery rate);
- коэффициент открытий (Open rate);
- коэффициент кликов по открытым письмам (Click-to-Open Rate, CTOR);
- коэффициент отписок (Unsubscribe rate).

## Ожидаемый результат
На основе рассчитанных метрик будет определена наиболее эффективная тема письма, а также сформулированы краткие выводы по результатам анализа.

In [2]:
#импор библиотек
import pandas as pd
import numpy as np

In [3]:
#загрузка данных
sheet_id = "1nu_aDq5dv2qa2rb89JJ9fy_OvURUJ5tBNlw_QcTLSnE"
sheet_name = "Data"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)
df.head()

,Название рассылки,Название кампании,Направление,Месяц,Дата,Год,Номер недели,День недели,День недели.1,Время,...,Доставлено,Открытия,Клики,Баунсы (Все ошибки),Отписки,UTM Метка,Пользователей на сайте,Воронка продаж. Шаг 1,Воронка продаж. Шаг 2,Воронка продаж. Шаг 3
0,Название рассылки 1,Название кампании 1,Email,Октябрь,27.10.2021,2021,43,3,03-среда,19:24,...,741 750,148 350,17 802,39 039,7 417,Метка 1,16 378,6 337,6 210,5 154
1,Название рассылки 10,Название кампании 10,Email,Ноябрь,05.11.2021,2021,45,5,05-пятница,12:02,...,683 402,123 012,11 071,35 969,6 834,Метка 10,10 296,3 558,3 096,2 539
2,Название рассылки 100,Название кампании 100,Email,Апрель,11.04.2022,2022,15,1,01-понедельник,16:26,...,1 141 344,182 615,15 340,60 071,11 413,Метка 100,13 959,3 370,2 864,2 263
3,Название рассылки 101,Название кампании 101,Email,Апрель,12.04.2022,2022,15,2,02-вторник,16:26,...,1 324 136,264 827,10 328,69 691,13 241,Метка 101,8 986,5 116,4 277,3 208
4,Название рассылки 102,Название кампании 102,Email,Апрель,13.04.2022,2022,15,3,03-среда,16:26,...,1 212 980,218 336,15 720,63 841,12 130,Метка 102,10 847,2 817,2 287,1 670


In [4]:
#Обзор данных
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Название рассылки       218 non-null    object
 1   Название кампании       218 non-null    object
 2   Направление             218 non-null    object
 3   Месяц                   218 non-null    object
 4   Дата                    218 non-null    object
 5   Год                     218 non-null    int64 
 6   Номер недели            218 non-null    int64 
 7   День недели             218 non-null    int64 
 8   День недели.1           218 non-null    object
 9   Время                   218 non-null    object
 10  Веб-версия              218 non-null    object
 11  Тема письма             218 non-null    object
 12  Сегмент                 218 non-null    object
 13  Отправлено              218 non-null    object
 14  Доставлено              218 non-null    object
 15  Открыт

Данные загрузились в полном объеме. Не все данные приведены к нужным типам, поправим ниже для необходимых колонок.

In [6]:
# убираем пробелы в колонках
df.columns = (
    df.columns
      .str.strip()                 
      .str.replace("\xa0", " ", regex=False)
)

In [7]:
#приведем нужные колонки к формату int, а время к datetime
numeric_cols = [
    "Отправлено",
    "Доставлено",
    "Открытия",
    "Клики",
    "Отписки"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .str.replace(r"\s+", "", regex=True)  # удаляем ВСЕ пробелы, включая \xa0
        .astype(int)
    )
#время к datetime
df["Дата"] = pd.to_datetime(df["Дата"], dayfirst=True)

df[numeric_cols + ["Дата"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Отправлено  218 non-null    int32         
 1   Доставлено  218 non-null    int32         
 2   Открытия    218 non-null    int32         
 3   Клики       218 non-null    int32         
 4   Отписки     218 non-null    int32         
 5   Дата        218 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int32(5)
memory usage: 6.1 KB


## Рачет метрик.

Считаем метрики классическим образом, как это принято в email-маркетинге:

- **Delivery Rate = количество доставленных писем / количество отправленных писем × 100%**

- **Open Rate = количество открытий / количество доставленных писем × 100%**

- **Click to Open Rate (CTOR) = количество кликов / количество открытий × 100%**

- **Unsubscribe Rate = количество отписок / количество доставленных писем × 100%**



In [9]:
#временная таблица для метрик
df_metrics = df[[
    "Дата",
    "Отправлено",
    "Доставлено",
    "Открытия",
    "Клики",
    "Отписки"
]].copy()

In [10]:
#переименуем колонку
df_metrics = df_metrics.rename(columns={"Дата": "date"})

In [11]:
#расчет метрик
df_metrics["delivery_rate"] = df_metrics["Доставлено"] / df_metrics["Отправлено"] * 100
df_metrics["open_rate"] = df_metrics["Открытия"] / df_metrics["Доставлено"] * 100
df_metrics["ctor"] = df_metrics["Клики"] / df_metrics["Открытия"] * 100
df_metrics["unsubscribe_rate"] = df_metrics["Отписки"] / df_metrics["Доставлено"] * 100

df_metrics[[
    "date",
    "delivery_rate",
    "open_rate",
    "ctor",
    "unsubscribe_rate"
]].round(2).head()

,date,delivery_rate,open_rate,ctor,unsubscribe_rate
0,2021-10-27,95.0,20.0,12.0,1.0
1,2021-11-05,95.0,18.0,9.0,1.0
2,2022-04-11,95.0,16.0,8.4,1.0
3,2022-04-12,95.0,20.0,3.9,1.0
4,2022-04-13,95.0,18.0,7.2,1.0


In [12]:
#посмотрим на данные с помощью описательной статистики
df_metrics.describe()

,date,Отправлено,Доставлено,Открытия,Клики,Отписки,delivery_rate,open_rate,ctor,unsubscribe_rate
count,218,2.180000e+02,2.180000e+02,218.000000,218.000000,218.000000,218.000000,218.000000,218.000000,218.000000
mean,2021-12-01 15:57:47.889908224,1.476656e+06,1.442694e+06,196476.197248,15915.105505,37804.871560,97.646791,13.763764,8.123433,2.470134
min,2021-04-15 00:00:00,5.100340e+05,4.998330e+05,50529.000000,1971.000000,1036.000000,94.999945,9.199971,3.899182,0.149942
25%,2021-10-30 18:00:00,9.304825e+05,9.124190e+05,127877.750000,9626.000000,5504.250000,97.999946,11.049988,7.199915,0.569994
50%,2021-12-11 12:00:00,1.497056e+06,1.454843e+06,177669.000000,13620.000000,9192.000000,97.999997,13.990011,8.400036,0.630006
75%,2022-01-25 00:00:00,1.988811e+06,1.943932e+06,256646.000000,19962.000000,15963.000000,98.499947,17.430002,9.000221,0.900010
max,2022-05-20 00:00:00,2.492076e+06,2.441119e+06,436664.000000,47292.000000,303772.000000,98.500077,20.000000,12.000624,13.000037
std,NaN,5.874587e+05,5.757326e+05,91896.220623,9326.534152,70871.015693,1.081472,3.391906,2.630963,4.173633


В целом, распределения ключевых метрик являются стабильнымими и находятся в ожидаемых для email-маркетинга диапазонах.
При этом отмечу, что по UR выявлен максимум около 13% при медианном значении менее 1%. Рекомендуется данную аномалию дополнительно проанализировать на примере конкретных тем писем, т.к. ограничен предоставленной информацией.

## Критерий выбора лучшей темы письма

В качестве основной метрики выбран показатель `Open Rate`, т.к. именно тема письма влияет на решение пользователя открыть письмо.
Дополнительно будем использовать метрику `Unsubscribe Rate` в качестве защитной, чтобы исключить темы, которые привлекают внимание, но вызывают негативную реакцию у пользователей.


In [15]:
#агрегация событий по каждой теме 
topic_stats = (
    df
    .groupby('Тема письма')
    .agg(
        delivered=('Доставлено', 'sum'),
        opens=('Открытия', 'sum'),
        clicks=('Клики', 'sum'),
        unsubscribes=('Отписки', 'sum')
    )
    .reset_index()
)

topic_stats.head()

,Тема письма,delivered,opens,clicks,unsubscribes
0,Тема письма 1,741750,148350,17802,7417
1,Тема письма 10,683402,123012,11071,6834
2,Тема письма 100,1141344,182615,15340,11413
3,Тема письма 101,1324136,264827,10328,13241
4,Тема письма 102,1212980,218336,15720,12130


In [16]:
#расчет метрик по темам
topic_stats['open_rate'] = topic_stats['opens'] / topic_stats['delivered']
topic_stats['unsubscribe_rate'] = topic_stats['unsubscribes'] / topic_stats['delivered']
topic_stats['ctor'] = topic_stats['clicks'] / topic_stats['opens']

In [17]:
#сортировка по Open Rate
topic_stats_sorted = topic_stats.sort_values(
    by='open_rate',
    ascending=False
)

topic_stats_sorted.head()

,Тема письма,delivered,opens,clicks,unsubscribes,open_rate,unsubscribe_rate,ctor
0,Тема письма 1,741750,148350,17802,7417,0.200000,0.009999,0.120000
3,Тема письма 101,1324136,264827,10328,13241,0.200000,0.010000,0.038999
81,Тема письма 172,545637,98215,8839,3110,0.180001,0.005700,0.089996
127,Тема письма 213,860236,154843,13007,1290,0.180001,0.001500,0.084001
52,Тема письма 146,561810,101126,8495,3202,0.180000,0.005699,0.084004


In [18]:
#добавляем отписки.
avg_ur = topic_stats['unsubscribe_rate'].mean()

best_topic = topic_stats_sorted[
    topic_stats_sorted['unsubscribe_rate'] <= avg_ur
].iloc[0]

best_topic

Тема письма         Тема письма 1
delivered                  741750
opens                      148350
clicks                      17802
unsubscribes                 7417
open_rate                     0.2
unsubscribe_rate         0.009999
ctor                         0.12
Name: 0, dtype: object

In [19]:
print(f"Лучшая тема письма — «{best_topic['Тема письма']}»")

Лучшая тема письма — «Тема письма 1»


# Вывод

В рамках исследования были рассчитаны ключевые метрики эффективности email-рассылок:

- Delivery Rate;
- Open Rate;
- Click-to-Open Rate;
- Unsubscribe Rate.

Отмечу, что `Unsubscribe Rate` в целом остается на низком уровне, однако для отдельных тем наблюдаются выбросы.

Для определения лучшей темы письма в качестве основной метрики был выбран Open Rate.
По результатам анализа лучшую динамику показала тема **«Тема письма 1»**, т.к. у нее наивысший Open Rate при приемлемом уровне отписок, что позволяет считать ее наиболее эффективной с точки зрения привлечения внимания пользователей.